# AI Sandbox: PDF -> Keyword Extraction -> DB Query Execution
This notebook demonstrates the pipeline using `litellm` and `instructor`.

## Observability + AI Platform Setup

Initialize **Arize AX** tracing and **LangChain / LangGraph** instrumentation.
This cell must run **before** any LLM clients are created.

In [1]:
import os
import grpc
from opentelemetry.sdk.trace import TracerProvider, Resource
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from openinference.instrumentation.langchain import LangChainInstrumentor

# env vars are injected by docker-compose env_file — no load_dotenv needed

if "tracer_provider" not in globals():
    space_id = os.environ["ARIZE_SPACE_ID"]
    api_key  = os.environ["ARIZE_API_KEY"]

    resource = Resource.create({
        "openinference.project.name": "fair-play-initiative-2",
    })

    exporter = OTLPSpanExporter(
        endpoint="otlp.arize.com",
        headers=[
            ("authorization", api_key),
            ("api_key",       api_key),
            ("arize-space-id", space_id),
            ("space_id",       space_id),
            ("arize-interface", "otel"),
        ],
        credentials=grpc.ssl_channel_credentials(),
    )

    tracer_provider = TracerProvider(resource=resource)
    tracer_provider.add_span_processor(SimpleSpanProcessor(exporter))

    LangChainInstrumentor().instrument(tracer_provider=tracer_provider)
    print("Arize AX tracing initialized — project: fair-play-initiative")
    print("Spans export immediately (SimpleSpanProcessor)")
else:
    print("Arize AX already initialized — skipping")

print("View traces → https://app.arize.com")

Arize AX tracing initialized — project: fair-play-initiative
Spans export immediately (SimpleSpanProcessor)
View traces → https://app.arize.com


In [2]:

# ── Observability: Local Metrics Tracker ─────────────────────────────────────
# Captures token usage, latency, and errors for every LLM call in this session.
# Runs alongside Arize AX traces for in-notebook inspection.

import time
from dataclasses import dataclass
from datetime import datetime
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.outputs import LLMResult


@dataclass
class RunMetrics:
    run_id: str
    timestamp: str
    latency_ms: float
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int
    error: str | None = None


session_metrics: list[RunMetrics] = []


class MetricsCallback(BaseCallbackHandler):
    """Captures latency + token usage for every LLM call."""

    def __init__(self):
        self._t: dict[str, float] = {}

    def on_llm_start(self, serialized, prompts, *, run_id, **kwargs):
        self._t[str(run_id)] = time.perf_counter()

    def on_llm_end(self, response: LLMResult, *, run_id, **kwargs):
        ms = (time.perf_counter() - self._t.pop(str(run_id), 0.0)) * 1000
        usage = (response.llm_output or {}).get("token_usage", {})
        m = RunMetrics(
            run_id=str(run_id),
            timestamp=datetime.now().isoformat(timespec="seconds"),
            latency_ms=round(ms, 1),
            prompt_tokens=usage.get("prompt_tokens", 0),
            completion_tokens=usage.get("completion_tokens", 0),
            total_tokens=usage.get("total_tokens", 0),
        )
        session_metrics.append(m)
        print(
            f"  [obs] {m.timestamp} | latency={m.latency_ms:.0f}ms"
            f" | tokens={m.total_tokens}"
            f" (prompt={m.prompt_tokens} / compl={m.completion_tokens})"
        )

    def on_llm_error(self, error, *, run_id, **kwargs):
        ms = (time.perf_counter() - self._t.pop(str(run_id), 0.0)) * 1000
        session_metrics.append(
            RunMetrics(
                run_id=str(run_id),
                timestamp=datetime.now().isoformat(timespec="seconds"),
                latency_ms=round(ms, 1),
                prompt_tokens=0,
                completion_tokens=0,
                total_tokens=0,
                error=str(error),
            )
        )
        print(f"  [obs] ERROR after {ms:.0f}ms — {error}")


metrics_cb = MetricsCallback()
print("MetricsCallback ready — attach via config={'callbacks': [metrics_cb]}")


MetricsCallback ready — attach via config={'callbacks': [metrics_cb]}


In [3]:
import os
import json
import fitz  # PyMuPDF
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
import sqlite3

In [4]:
# ── Load Schema + System Prompt from files ────────────────────────────────────
with open("Schema.json") as f:
    FPI_SCHEMA = json.load(f)

with open("SystemPrompt.md") as f:
    KEYWORD_SYSTEM_PROMPT = f.read()

# Build full system message: prompt text + schema JSON appended
_schema_str = json.dumps(FPI_SCHEMA, indent=2)
KEYWORD_SYSTEM_MSG = f"{KEYWORD_SYSTEM_PROMPT}\n\n# FPI Schema\n\n```json\n{_schema_str}\n```"

print(f"Schema loaded: {len(FPI_SCHEMA)} tables — {[t['table'] for t in FPI_SCHEMA]}")
print(f"System prompt: {len(KEYWORD_SYSTEM_PROMPT)} chars")

Schema loaded: 9 tables — ['region', 'organization', 'organization_region', 'policy', 'rule', 'employee', 'point_history', 'attendance_log', 'alert']
System prompt: 4164 chars


## 1. Define Output Schemas
Two Pydantic models define the structured output for each pipeline step: **keyword extraction** (`KeywordExtraction`) and **SQL generation** (`SQLGeneration`). They are kept in separate cells so each step can be run and tested independently.

In [5]:
class KeywordExtraction(BaseModel):
    policy: list[str] = Field(default_factory=list, description="Policy-level keywords: codes, names, framework attributes, accrual model type, and scope.")
    organization: list[str] = Field(default_factory=list, description="Organization names and identifiers.")
    region: list[str] = Field(default_factory=list, description="Region names, jurisdictions, and applicable labor laws.")
    rule: list[str] = Field(default_factory=list, description="Rule names, tier definitions, conditions, point thresholds, and escalation criteria.")
    attendance_log: list[str] = Field(default_factory=list, description="Attendance event types, violation definitions, scheduled vs actual time terms, and log statuses.")
    point_history: list[str] = Field(default_factory=list, description="Point accrual, expiration, active status, and reason classification terms.")
    alert: list[str] = Field(default_factory=list, description="Alert types, automatic trigger conditions, escalation actions, and documentation levels.")
    other_relevant_terms: list[str] = Field(default_factory=list, description="Terms relevant to the policy but not directly mappable to a specific schema element.")
    confidence_score: float = Field(description="Confidence score of the extraction between 0.0 and 1.0.")

In [6]:
class SQLGeneration(BaseModel):
    sql_query: str = Field(description="SQL query generated based on extracted keywords, targeting the FPI schema tables.")
    confidence_score: float = Field(description="Confidence score of the SQL generation between 0.0 and 1.0.")

## 1b. SQL Query Plan Schema

Sub-models and `SQLQueryPlan` define the structured decomposition of a query
before any SQL is written. The plan is reviewed in Arize AX before SQL generation proceeds.

In [7]:
# ── SQL Query Plan: Decomposition Sub-models ──────────────────────────────────
# Top-level classes required for clean JSON Schema $defs with with_structured_output.

class TableUsage(BaseModel):
    table: str = Field(description="Exact table name from the FPI schema.")
    alias: str = Field(description="Short alias used in the SQL query (e.g. 'e' for employee).")
    purpose: str = Field(description="One sentence: why this table is needed for this query.")

class JoinStep(BaseModel):
    left: str = Field(description="Left-side table alias or name.")
    right: str = Field(description="Right-side table alias or name.")
    condition: str = Field(description="The ON clause expression, e.g. 'e.organization_id = o.id'.")
    join_type: str = Field(description="JOIN type: INNER, LEFT, RIGHT, or FULL.")

class FilterCondition(BaseModel):
    description: str = Field(description="Plain-English description of what this filter does.")
    expression: str = Field(description="The SQL expression for this filter.")
    source_keyword: str = Field(description="The extracted keyword that motivated this filter.")

class AggregationStep(BaseModel):
    column_name: str = Field(description="Output column alias for this aggregation.")
    expression: str = Field(description="The aggregate SQL expression.")
    purpose: str = Field(description="What business question this aggregation answers.")

class SQLQueryPlan(BaseModel):
    objective: str = Field(
        description="One sentence: what does this SQL operation accomplish? "
                    "For ingestion plans, describe what data is being inserted and into which tables. "
                    "Do NOT describe downstream analytics or reporting use cases."
    )
    tables_required: list[TableUsage] = Field(
        default_factory=list,
        description="All tables needed, with aliases and purpose.",
    )
    joins: list[JoinStep] = Field(
        default_factory=list,
        description="All JOIN relationships required, in the order they should be applied.",
    )
    where_filters: list[FilterCondition] = Field(
        default_factory=list,
        description="All WHERE-clause filters, each tied to a source keyword.",
    )
    grouping_columns: list[str] = Field(
        default_factory=list,
        description="Column references to include in GROUP BY. Leave empty for ingestion plans.",
    )
    aggregations: list[AggregationStep] = Field(
        default_factory=list,
        description="All aggregate columns to compute. Leave empty for ingestion plans.",
    )
    having_filters: list[FilterCondition] = Field(
        default_factory=list,
        description="Post-aggregation HAVING filters. Leave empty for ingestion plans.",
    )
    output_columns: list[str] = Field(
        default_factory=list,
        description="Final SELECT column names or aliases. Leave empty for ingestion plans.",
    )
    uses_cte: bool = Field(
        description="True if the query requires a Common Table Expression (WITH clause)."
    )
    cte_description: str | None = Field(
        default=None,
        description="If uses_cte is True, describe what the CTE computes and why.",
    )
    confidence_score: float = Field(
        description="Confidence in the plan's correctness and completeness, 0.0–1.0."
    )

print("SQLQueryPlan schema defined — 5 models (TableUsage, JoinStep, FilterCondition, AggregationStep, SQLQueryPlan)")

SQLQueryPlan schema defined — 5 models (TableUsage, JoinStep, FilterCondition, AggregationStep, SQLQueryPlan)


## 2. PDF Extraction using PyMuPDF
We use `pymupdf` (fitz) to extract the full text from the SilverPeak Fabrication attendance policy PDF.

In [8]:
#PDF_PATH = "/input_files/HR-811-A_SilverPeak_Fabrication_Attendance_Policy_Complex.pdf"
PDF_PATH = "/input_files/Acme_Manufacturing_Attendance_Policy.pdf"

doc = fitz.open(PDF_PATH)
num_pages = len(doc)
pdf_text = ""
for page in doc:
    pdf_text += page.get_text()
doc.close()

print(f"Extracted {num_pages} pages, {len(pdf_text)} characters")
print("\n--- First 600 characters ---")
print(pdf_text[:600])


Extracted 8 pages, 15239 characters

--- First 600 characters ---
ACME MANUFACTURING CORPORATION
Employee Attendance & Punctuality Policy
Policy Number: HR-ATT-2024-001   |   Effective Date: January 1, 2024   |   Revision: 3.0
1. Purpose
This policy establishes uniform standards for employee attendance and punctuality across all Acme
Manufacturing Corporation (AMC) facilities. Consistent attendance is critical to maintaining safe
production operations, meeting customer commitments, and supporting co-workers who depend on
each team member's presence. This policy applies to all hourly production employees, skilled trades,
quality technicians, warehouse associa


## 3. Keyword Extraction (Step 3a)
Uses `langchain` + `with_structured_output` with `KeywordExtraction`. The system prompt and FPI schema are loaded from `SystemPrompt.md` and `Schema.json`.

**Step 3b — SQL Generation** is in the next cell and is not yet tested.

In [9]:
# ── Model Setup ───────────────────────────────────────────────────────────────
from langchain_core.messages import SystemMessage
from langchain_core.prompts import HumanMessagePromptTemplate

# Keyword extraction + SQL generation: gpt-5-mini (fast, cheap, structured output)
model = init_chat_model(
    model="gpt-5-mini",
    model_provider="openai",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENAI_API_KEY"],
)

# Query plan / explainability: claude-sonnet-4-6 (stronger reasoning for join + filter logic)
plan_model_base = init_chat_model(
    model="gpt-5-mini",
    model_provider="openai",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENAI_API_KEY"],
)

kw_model = model.with_structured_output(KeywordExtraction)

KEYWORD_PROMPT = ChatPromptTemplate.from_messages([
    SystemMessage(content=KEYWORD_SYSTEM_MSG),
    HumanMessagePromptTemplate.from_template("{pdf_text}"),
])

_KEYWORD_FIELDS = [
    "policy", "organization", "region", "rule",
    "attendance_log", "point_history", "alert", "other_relevant_terms",
]


def _format_keywords(kw: KeywordExtraction) -> str:
    """Format a KeywordExtraction into a readable string for downstream prompts."""
    lines = []
    for field in _KEYWORD_FIELDS:
        values = getattr(kw, field, [])
        if values:
            lines.append(f"[{field}]")
            lines.extend(f"  • {v}" for v in values)
    return "\n".join(lines)


# ── Run: Step 3a — Keyword Extraction ────────────────────────────────────────
try:
    keyword_data = (KEYWORD_PROMPT | kw_model).invoke({"pdf_text": pdf_text})
    print("--- Keyword Extraction Results ---")
    for field in _KEYWORD_FIELDS:
        values = getattr(keyword_data, field, [])
        if values:
            print(f"\n[{field}] ({len(values)} terms):")
            for kw in values:
                print(f"  • {kw}")
    print(f"\nConfidence: {keyword_data.confidence_score}")
except Exception as e:
    print(f"Error: {e}")
    keyword_data = None

--- Keyword Extraction Results ---

[policy] (10 terms):
  • HR-ATT-2024-001
  • Effective Date: 2024-01-01
  • Revision 3.0
  • Attendance & Punctuality Policy
  • Applies to hourly production, skilled trades, warehouse, QA, supervisors, temps/contractors
  • Excludes salaried exempt (see HR-SAL-2024-002)
  • Acknowledgment required via HRIS within 5 business days
  • Policy reviewed annually by VP HR & Director Labor Relations
  • Records retained 5 years
  • Policy changes posted 15 days before effective date

[organization] (8 terms):
  • Acme Manufacturing Corporation (AMC)
  • Organization code: AMC
  • Active organization
  • HR contact hr@acmemfg.com
  • HR phone ext. 2200
  • Production Control Hotline ext. 1100
  • Weather Hotline ext. 9000
  • Ethics Hotline 1-800-AMC-SAFE

[region] (7 terms):
  • All AMC facilities
  • Region labor laws: FMLA
  • Region labor laws: ADA
  • Region labor laws: Workers' Compensation
  • Region labor laws: USERRA (military)
  • Applicable feder

/usr/local/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=KeywordExtraction(policy=..., confidence_score=0.95), input_type=KeywordExtraction])
  return self.__pydantic_serializer__.to_python(


### Step 3b - Query Plan Generation (Decomposition)

The LLM devises a **structured plan** — which tables, joins, filters, and aggregations to use —
**without writing any SQL**. This plan is the authoritative specification for Step 3c.
Full plan JSON is captured as a span in Arize AX for review.

In [10]:
# ── Plan Prompt Setup ─────────────────────────────────────────────────────────
PLAN_SYSTEM_MSG = (
    "You are a SQL ingestion planning assistant for the Fair Play Initiative (FPI). "
    "Your task is to analyse extracted policy keywords and devise a precise, "
    "structured plan for INSERT/UPSERT statements that load this policy document "
    "into the FPI database.\n\n"
    "WHAT YOU ARE PLANNING: policy document ingestion — not analytics or reporting. "
    "The goal is to persist the organisation, region, policy metadata, and rules "
    "extracted from the PDF into the appropriate FPI configuration tables.\n\n"
    "TARGET TABLES (the only tables this plan may touch):\n"
    "  • organization  — one row for the employer entity\n"
    "  • region        — one row for the jurisdiction\n"
    "  • organization_region — junction row linking the two\n"
    "  • policy        — one row for the policy document itself\n"
    "  • rule          — one row per violation tier / escalation rule\n\n"
    "OFF-LIMITS TABLES (do NOT plan any writes to these):\n"
    "  • employee, attendance_log, point_history, alert\n"
    "  These are populated at runtime by the HR system, never by policy ingestion.\n\n"
    "CRITICAL CONSTRAINT: Do NOT write any SQL. Do NOT include SQL fragments in "
    "any field. Your entire output must be a structured plan in plain English "
    "and column-name notation only.\n\n"
    # ── Distillation notes ────────────────────────────────────────────────────
    # The following guidance was derived from expert model reasoning on this task.
    # Follow it precisely.
    "PLANNING GUIDANCE\n\n"
    "INSERT order — always respect FK constraints:\n"
    "  organization → region → organization_region → policy → rule\n\n"
    "INSERT strategy per table:\n"
    "  • organization        : INSERT OR IGNORE — master record; re-ingestion must not overwrite\n"
    "  • region              : INSERT OR IGNORE — shared infrastructure; same reason\n"
    "  • organization_region : INSERT OR IGNORE — junction row; no column data to overwrite\n"
    "  • policy              : INSERT OR REPLACE — a revised policy document supersedes prior versions; "
    "effective_date and active flag must stay current\n"
    "  • rule                : INSERT OR REPLACE — policy revisions may update thresholds and point values\n\n"
    "ID chaining — always set uses_cte = True and describe the full chain in cte_description:\n"
    "  After each INSERT, the generated ID flows as FK into subsequent INSERTs:\n"
    "  organization.id → region.id → organization_region.(organization_id, region_id) "
    "→ policy.(organization_id, region_id) → policy.id → rule.policy_id\n"
    "  Without this chain, FK constraints cannot be satisfied.\n\n"
    "cte_description — this field carries the COMPLETE procedural plan. "
    "It MUST contain all of the following (gpt-5-mini: do not omit any section):\n"
    "  1. For each table in INSERT order, a numbered step containing:\n"
    "     - Table name and INSERT strategy (OR IGNORE / OR REPLACE)\n"
    "     - Every column to be populated, mapped to the exact extracted keyword that provides its value\n"
    "     - The natural key used for the existence check (matches the where_filters entry)\n"
    "  2. A complete enumeration of ALL rule rows in four groups — include every row "
    "from the policy; do not summarise or omit:\n"
    "     a. Violation/occurrence rules — one row per violation type (points > 0)\n"
    "     b. Approved-leave rules — one row per leave type (points = 0.0)\n"
    "     c. Perfect-attendance reduction rules — one row per milestone "
    "(threshold = consecutive clean days: 30, 60, 90, 365; points = negative value)\n"
    "     d. Corrective-action trigger rules — one row per level "
    "(threshold = lower bound of point range; points = 0.0)\n"
    "     Also include operational rule rows for: rolling 12-month window, negative balance floor, "
    "occurrence definition, excused/unexcused determination deadlines.\n"
    "     For each rule row provide: id (kebab slug), name, condition (plain-English HRIS trigger), "
    "threshold, points, description, active.\n"
    "  3. ID chaining summary (one line per FK relationship).\n"
    "  4. Operational reminders — placeholders that need resolution from HRIS/facility master data "
    "(region code, timezone, mandatory OT points confirmation, etc.).\n"
    "  Completeness is critical — a missing rule row means the policy is partially loaded.\n\n"
    "joins — for ingestion plans, use this field to describe FK dependency chains, NOT SQL joins:\n"
    "  • left: the child table (the one holding the FK column)\n"
    "  • right: the parent table (the one holding the PK)\n"
    "  • condition: which FK in the child references which PK in the parent "
    "(e.g. 'organization_region.organization_id = organization.id')\n"
    "  • join_type: always 'INNER' for required FK dependencies\n"
    "  List one entry per FK relationship, in dependency order.\n\n"
    "Rule rows — plan one row per:\n"
    "  • Each distinct violation type (NCNS 1st day, NCNS consecutive, unexcused full-day, "
    "unexcused half-day, tardy major, tardy minor, third minor tardy in month, "
    "early departure, holiday/critical day absence, plant shutdown no-return, "
    "mandatory OT no-show, voluntary OT no-show, and any other named in the policy)\n"
    "  • Each approved leave type that generates 0.0 points "
    "(FMLA, ADA, PTO/vacation, jury duty, military/USERRA, bereavement, workers comp, STD, "
    "and any other named in the policy)\n"
    "  • Each perfect attendance reduction milestone "
    "(threshold = consecutive clean days: 30, 60, 90, 365; points = negative value)\n"
    "  • Each corrective action level "
    "(threshold = lower bound of the point range; points = 0.0 — these are trigger rules, "
    "not point-generating occurrences)\n"
    "  Do NOT merge multiple violation types into a single rule row.\n\n"
    "Value derivation:\n"
    "  • id (organization, region, policy): generate a lower-kebab-case slug "
    "(e.g. 'acme-manufacturing', 'hr-att-2024-001')\n"
    "  • region.code: if not stated explicitly in the policy, derive a placeholder "
    "(e.g. 'AMC-PLANT-US') and note it must be resolved from facility master data\n"
    "  • region.timezone: infer from shift schedule context or mark as a placeholder "
    "(e.g. 'America/Detroit') to be confirmed from HRIS\n"
    "  • region.labor_laws: collect all referenced statutes as comma-separated TEXT "
    "(e.g. 'FMLA, ADA, USERRA, Workers Compensation')\n"
    "  • rule.condition: describe the HRIS/system condition that triggers this rule "
    "in plain English — do not write SQL\n"
    "  • rule.threshold: occurrence rules = 1; corrective action levels = lower bound "
    "of the point range; perfect attendance = consecutive clean days\n"
    "  • rule.points: positive for violations; 0.0 for approved-leave and CA-trigger rows; "
    "NEGATIVE for perfect-attendance reductions (e.g. -0.5, -1.0, -1.5, -2.0)\n\n"
    "Natural keys for existence checks (where_filters):\n"
    "  • organization: name OR code\n"
    "  • region: code\n"
    "  • organization_region: composite (organization_id, region_id)\n"
    "  • policy: name AND organization_id\n"
    "  • rule: policy_id AND name\n\n"
    "Planning rules:\n"
    "1. List tables_required in INSERT order (organization first, rule last).\n"
    "2. List joins as FK dependency chains: child table (left) → parent table (right); "
    "condition = FK column = PK column; join_type = 'INNER'.\n"
    "3. List where_filters: one existence check per table using the natural keys above. "
    "Set source_keyword to the extracted keyword that provides the natural key value.\n"
    "4. Put ALL column-to-keyword mappings and ALL rule row enumerations in cte_description. "
    "Do NOT put column mappings only in the purpose field of tables_required.\n"
    "5. Leave grouping_columns, aggregations, having_filters, and output_columns empty.\n\n"
    f"# FPI Schema\n\n```json\n{json.dumps(FPI_SCHEMA, indent=2)}\n```"
)

plan_model = model.with_structured_output(SQLQueryPlan)

PLAN_PROMPT = ChatPromptTemplate.from_messages([
    SystemMessage(content=PLAN_SYSTEM_MSG),
    HumanMessagePromptTemplate.from_template(
        "Devise an ingestion plan for the following keywords extracted from the "
        "attendance policy document. The plan must describe how to INSERT this "
        "policy's data into the FPI configuration tables. Do NOT write SQL.\n\n"
        "## Extracted Keywords\n\n{keywords}\n\n"
        "## Policy Document (for context)\n\n{pdf_text}"
    ),
])

print("PLAN_PROMPT ready — plan_model configured with SQLQueryPlan structured output")


PLAN_PROMPT ready — plan_model configured with SQLQueryPlan structured output


In [11]:
# ── _format_plan: plain-text serialiser for SQLQueryPlan ─────────────────────
def _format_plan(plan: SQLQueryPlan) -> str:
    lines = []
    lines.append(f"OBJECTIVE\n  {plan.objective}\n")

    if plan.tables_required:
        lines.append("TABLES")
        for t in plan.tables_required:
            lines.append(f"  [{t.alias}] {t.table}")
            lines.append(f"       purpose: {t.purpose}")
        lines.append("")

    if plan.joins:
        lines.append("JOINS")
        for j in plan.joins:
            lines.append(f"  {j.join_type} JOIN {j.right} ON {j.condition}")
        lines.append("")

    if plan.where_filters:
        lines.append("WHERE FILTERS")
        for f in plan.where_filters:
            lines.append(f"  • {f.description}")
            lines.append(f"    expr   : {f.expression}")
            lines.append(f"    keyword: {f.source_keyword}")
        lines.append("")

    if plan.grouping_columns:
        lines.append("GROUP BY")
        lines.append("  " + ", ".join(plan.grouping_columns))
        lines.append("")

    if plan.aggregations:
        lines.append("AGGREGATIONS")
        for a in plan.aggregations:
            lines.append(f"  [{a.column_name}]")
            lines.append(f"    expr   : {a.expression}")
            lines.append(f"    purpose: {a.purpose}")
        lines.append("")

    if plan.having_filters:
        lines.append("HAVING FILTERS")
        for f in plan.having_filters:
            lines.append(f"  • {f.description}")
            lines.append(f"    expr   : {f.expression}")
            lines.append(f"    keyword: {f.source_keyword}")
        lines.append("")

    if plan.output_columns:
        lines.append("OUTPUT COLUMNS")
        lines.append("  " + ", ".join(plan.output_columns))
        lines.append("")

    if plan.uses_cte:
        lines.append("CTE")
        lines.append(f"  {plan.cte_description or '(no description)'}")

    return "\n".join(lines)


# ── Run: Step 3b — Query Plan Generation ─────────────────────────────────────
plan_data: SQLQueryPlan | None = None

if keyword_data:
    try:
        plan_data = (PLAN_PROMPT | plan_model).invoke(
            {
                "keywords": _format_keywords(keyword_data),
                "pdf_text": pdf_text,
            },
            config={"callbacks": [metrics_cb]},
        )
        print(_format_plan(plan_data))
        print(f"Confidence: {plan_data.confidence_score}")
    except Exception as e:
        print(f"Plan generation error: {e}")
        plan_data = None
else:
    print("Run Step 3a first (keyword_data is None).")

/usr/local/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=SQLQueryPlan(objective='I..., confidence_score=0.87), input_type=SQLQueryPlan])
  return self.__pydantic_serializer__.to_python(


  [obs] 2026-03-05T12:27:06 | latency=197260ms | tokens=21193 (prompt=8598 / compl=12595)
OBJECTIVE
  Ingest the Acme Manufacturing Corporation Attendance & Punctuality Policy (HR-ATT-2024-001) into FPI configuration tables: create organization, region, organization_region link, upsert policy metadata, and upsert all policy rule rows (occurrence rules, approved-leave rules, perfect-attendance reductions, corrective-action triggers, and operational rules).

TABLES
  [o] organization
       purpose: Master employer record for Acme Manufacturing Corporation (used as PK for policy and region link).
  [r] region
       purpose: Jurisdictional/operational scope record for 'All AMC facilities' with labor laws and timezone placeholder for facility master resolution.
  [or] organization_region
       purpose: Junction linking Acme Manufacturing Corporation to the 'All AMC facilities' region.
  [p] policy
       purpose: Policy document row for HR-ATT-2024-001 containing effective date, active f

In [ ]:
# ── Step 3c — Plan-Guided SQL Generation ──────────────────────────────────────
# Runs after Step 3b. SQL is strictly constrained by the approved ingestion plan.
# The model generates INSERT/UPSERT statements — it does not add or remove tables/filters.

PLAN_GUIDED_SQL_SYSTEM_MSG = (
    "You are a SQL generation assistant for the Fair Play Initiative (FPI). "
    "You will receive two inputs: (1) an approved ingestion plan that specifies "
    "exactly which configuration tables to INSERT into and how, and (2) keywords "
    "extracted from the policy document that supply the literal values.\n\n"
    "Your task: implement the plan as INSERT OR IGNORE / INSERT OR REPLACE "
    "statements targeting the FPI SQLite schema. The plan is the authoritative "
    "specification.\n"
    "  • Do NOT generate SELECT, UPDATE, or DELETE statements.\n"
    "  • Do NOT touch employee, attendance_log, point_history, or alert tables.\n"
    "  • Do NOT add tables or columns that are not in the plan.\n"
    "  • Do NOT remove any INSERT that the plan requires.\n\n"
    "INSERT order must respect FK constraints:\n"
    "  organization → region → organization_region → policy → rule\n\n"
    "SQLite compatibility rules:\n"
    "  • Use INSERT OR IGNORE when the plan says 'skip if exists'.\n"
    "  • Use INSERT OR REPLACE when the plan says 'overwrite if exists'.\n"
    "  • Generate a slug ID from the org/policy name when no natural key exists "
    "    (e.g. lower-kebab-case: 'acme-manufacturing' or 'hr-att-2024-001').\n"
    "  • Store labor_laws as a plain TEXT value (comma-separated list).\n"
    "  • Use ISO 8601 date strings for DATE columns (YYYY-MM-DD).\n"
    "  • Output all INSERTs as a single executable SQL block.\n\n"
    f"# FPI Schema\n\n```json\n{json.dumps(FPI_SCHEMA, indent=2)}\n```"
)

def _format_plan_for_sql_prompt(plan: SQLQueryPlan) -> str:
    """Structured plan format for the SQL generation model."""
    lines = [f"Objective: {plan.objective}", ""]

    if plan.tables_required:
        lines.append("Tables (in INSERT order):")
        for t in plan.tables_required:
            lines.append(f"  {t.table} AS {t.alias} — {t.purpose}")
        lines.append("")

    if plan.joins:
        lines.append("Joins:")
        for j in plan.joins:
            lines.append(f"  {j.join_type} JOIN {j.right} ON {j.condition}")
        lines.append("")

    if plan.where_filters:
        lines.append("Existence checks / WHERE:")
        for f in plan.where_filters:
            lines.append(f"  {f.description}: {f.expression}")
        lines.append("")

    if plan.output_columns:
        lines.append(f"Output columns: {', '.join(plan.output_columns)}")

    if plan.uses_cte and plan.cte_description:
        lines.append("")
        lines.append(f"CTE: {plan.cte_description}")

    return "\n".join(lines)

PLAN_GUIDED_SQL_PROMPT = ChatPromptTemplate.from_messages([
    SystemMessage(content=PLAN_GUIDED_SQL_SYSTEM_MSG),
    HumanMessagePromptTemplate.from_template(
        "Generate INSERT SQL implementing the ingestion plan below.\n\n"
        "## Approved Ingestion Plan\n\n{plan}\n\n"
        "## Extracted Keywords (source values for INSERT literals)\n\n{keywords}"
    ),
])

sql_model = model.with_structured_output(SQLGeneration)

# ── Run: Step 3c ──────────────────────────────────────────────────────────────
sql_data: SQLGeneration | None = None

if plan_data:
    try:
        sql_data = (PLAN_GUIDED_SQL_PROMPT | sql_model).invoke(
            {
                "plan": _format_plan_for_sql_prompt(plan_data),
                "keywords": _format_keywords(keyword_data),
            },
            config={"callbacks": [metrics_cb]},
        )
        print(f"SQL generated (confidence={sql_data.confidence_score:.2f}):\n")
        print(sql_data.sql_query)
    except Exception as e:
        print(f"SQL generation error: {e}")
        sql_data = None
else:
    print("Run Step 3b (plan generation) first.")

## 4. Schema Setup + SQL Validation
Create an in-memory SQLite DB with the full FPI schema (no seed data). The AI-generated SQL is then validated for syntax correctness against the empty schema. Data seeding happens in a later phase.

In [ ]:
conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

# --- FPI Schema (no seed data — controlled experiment) ---
cursor.executescript("""
    CREATE TABLE organizations (
        id   TEXT PRIMARY KEY,
        name TEXT,
        code TEXT,
        active BOOLEAN DEFAULT 1
    );

    CREATE TABLE regions (
        id         TEXT PRIMARY KEY,
        name       TEXT,
        code       TEXT,
        timezone   TEXT,
        labor_laws TEXT
    );

    CREATE TABLE policies (
        id              TEXT PRIMARY KEY,
        name            TEXT,
        organization_id TEXT REFERENCES organizations(id),
        region_id       TEXT REFERENCES regions(id),
        active          BOOLEAN DEFAULT 1,
        effective_date  DATE
    );

    CREATE TABLE rules (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        policy_id   TEXT REFERENCES policies(id),
        name        TEXT,
        condition   TEXT,
        threshold   INTEGER,
        points      REAL,
        description TEXT,
        active      BOOLEAN DEFAULT 1
    );

    CREATE TABLE employees (
        id              INTEGER PRIMARY KEY AUTOINCREMENT,
        first_name      TEXT,
        last_name       TEXT,
        email           TEXT UNIQUE,
        department      TEXT,
        position        TEXT,
        start_date      DATE,
        points          REAL DEFAULT 0,
        trend           TEXT DEFAULT 'stable',
        next_reset      DATE,
        organization_id TEXT REFERENCES organizations(id),
        region_id       TEXT REFERENCES regions(id),
        policy_id       TEXT REFERENCES policies(id)
    );

    CREATE TABLE attendance_logs (
        id              INTEGER PRIMARY KEY AUTOINCREMENT,
        employee_id     INTEGER REFERENCES employees(id),
        organization_id TEXT,
        region_id       TEXT,
        policy_id       TEXT,
        date            DATE,
        scheduled_in    TEXT,
        scheduled_out   TEXT,
        actual_in       TEXT,
        actual_out      TEXT,
        violation       TEXT,
        points          REAL DEFAULT 0,
        status          TEXT DEFAULT 'Compliant'
    );

    CREATE TABLE point_history (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        employee_id INTEGER REFERENCES employees(id),
        date        DATE,
        type        TEXT,
        points      REAL,
        status      TEXT DEFAULT 'Active',
        reason      TEXT
    );

    CREATE TABLE alerts (
        id              INTEGER PRIMARY KEY AUTOINCREMENT,
        organization_id TEXT,
        type            TEXT,
        message         TEXT,
        created_at      DATETIME DEFAULT CURRENT_TIMESTAMP
    );
""")

conn.commit()
print("Schema ready — 8 tables created, no seed data.")
print("Tables:", [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()])

# --- Validate AI-generated SQL (syntax check against empty schema) ---
if sql_data:
    try:
        print(f"\nValidating SQL:\n{sql_data.sql_query}\n")
        cursor.execute(sql_data.sql_query)
        rows = cursor.fetchall()
        col_names = [d[0] for d in cursor.description] if cursor.description else []
        print(f"Columns : {col_names}")
        print(f"Rows    : {len(rows)} (empty — no data seeded yet)")
    except Exception as e:
        print(f"\nSQL Error: {e}")
else:
    print("\nNo sql_data yet — run Step 3c to generate SQL.")

---
## LangGraph Agent Pipeline (Decomposition Edition)

A typed, observable graph that implements the three-stage decomposition approach.
Every node call is automatically captured as a span in **Arize AX**.

```
pdf_text → [extract] → [plan] → [generate_sql] → [execute] → query_results
```

- **extract**: `KeywordExtraction` — categorised terms from the PDF
- **plan**: `SQLQueryPlan` — tables, joins, filters, aggregations (no SQL yet)
- **generate_sql**: `SQLGeneration` — SQL strictly derived from the plan
- **execute**: runs SQL against in-memory SQLite → `query_results`

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

# Structured output model variants for graph nodes
lg_kw_model   = model.with_structured_output(KeywordExtraction)      # gpt-5-mini
lg_plan_model = model.with_structured_output(SQLQueryPlan)           # gpt-5-mini + distillation notes
lg_sql_model  = model.with_structured_output(SQLGeneration)          # gpt-5-mini

# ── Graph State ───────────────────────────────────────────────────────────────
class FPIState(TypedDict):
    pdf_text:      str
    keywords:      KeywordExtraction | None   # output of node_extract
    sql_plan:      SQLQueryPlan | None        # output of node_plan
    sql_result:    SQLGeneration | None       # output of node_generate_sql
    query_results: list[dict] | None          # output of node_execute
    error:         str | None

# ── Node: Extract Keywords ────────────────────────────────────────────────────
def node_extract(state: FPIState) -> dict:
    """Categorise policy keywords — traced automatically by Arize AX."""
    result = (KEYWORD_PROMPT | lg_kw_model).invoke(
        {"pdf_text": state["pdf_text"][:4000]},
    )
    return {"keywords": result}

# ── Node: Generate Query Plan ─────────────────────────────────────────────────
def node_plan(state: FPIState) -> dict:
    """Produce a structured SQLQueryPlan from keywords. No SQL is written here."""
    if not state.get("keywords"):
        return {"error": "node_plan: no keywords in state"}
    result = (PLAN_PROMPT | lg_plan_model).invoke(
        {
            "keywords": _format_keywords(state["keywords"]),
            "pdf_text": state["pdf_text"][:2000],
        },
    )
    return {"sql_plan": result}

# ── Node: Generate SQL from Plan ──────────────────────────────────────────────
def node_generate_sql(state: FPIState) -> dict:
    """Implement the approved plan as SQL. Constrained to plan spec."""
    if not state.get("sql_plan"):
        return {"error": "node_generate_sql: no sql_plan in state"}
    result = (PLAN_GUIDED_SQL_PROMPT | lg_sql_model).invoke(
        {
            "plan": _format_plan_for_sql_prompt(state["sql_plan"]),
            "keywords": _format_keywords(state["keywords"]),
        },
    )
    return {"sql_result": result}

# ── Node: Execute SQL ─────────────────────────────────────────────────────────
def node_execute(state: FPIState) -> dict:
    """Execute the generated SQL against the in-memory SQLite connection."""
    if not state.get("sql_result"):
        return {"error": "node_execute: no sql_result in state"}
    try:
        cur = conn.cursor()
        cur.execute(state["sql_result"].sql_query)
        rows = [dict(r) for r in cur.fetchall()]
        return {"query_results": rows}
    except Exception as exc:
        return {"error": str(exc)}

# ── Build & Compile Graph ─────────────────────────────────────────────────────
builder = StateGraph(FPIState)
builder.add_node("extract",      node_extract)
builder.add_node("plan",         node_plan)
builder.add_node("generate_sql", node_generate_sql)
builder.add_node("execute",      node_execute)

builder.set_entry_point("extract")
builder.add_edge("extract",      "plan")
builder.add_edge("plan",         "generate_sql")
builder.add_edge("generate_sql", "execute")
builder.add_edge("execute",      END)

fpi_agent = builder.compile()
print("LangGraph agent compiled: extract → plan → generate_sql → execute")

In [ ]:
# ── Run LangGraph Agent (Decomposition Pipeline) ─────────────────────────────
lg_result = fpi_agent.invoke(
    {
        "pdf_text":      pdf_text,
        "keywords":      None,
        "sql_plan":      None,
        "sql_result":    None,
        "query_results": None,
        "error":         None,
    },
    config={"callbacks": [metrics_cb]},
)

# ── Print results ─────────────────────────────────────────────────────────────
if lg_result.get("error"):
    print(f"Pipeline error: {lg_result['error']}")
else:
    kw  = lg_result["keywords"]
    pl  = lg_result["sql_plan"]
    sql = lg_result["sql_result"]

    kw_count = sum(len(getattr(kw, f, [])) for f in _KEYWORD_FIELDS) if kw else 0
    print(f"Keywords extracted : {kw_count} terms")
    print(f"Plan objective     : {pl.objective if pl else 'N/A'}")
    print(f"Plan tables        : {[t.table for t in pl.tables_required] if pl else []}")
    print(f"Plan confidence    : {pl.confidence_score if pl else 'N/A'}")
    print(f"SQL confidence     : {sql.confidence_score if sql else 'N/A'}")
    print(f"\nGenerated SQL:\n{sql.sql_query if sql else 'N/A'}")
    print(f"\nQuery rows : {len(lg_result.get('query_results') or [])}")

print("\nTrace exported → https://app.arize.com  (project: fair-play-initiative)")

In [ ]:
# ── Human Eval: Categorical Review Widget ────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display, clear_output

human_reviews: list[dict] = []


def _extraction_to_text(extraction) -> str:
    """Format categorized KeywordExtraction for display."""
    lines = []
    for field in _KEYWORD_FIELDS:
        values = getattr(extraction, field, [])
        if values:
            lines.append(f"[{field}]")
            lines.extend(f"  • {v}" for v in values)
    return "\n".join(lines)


def review_extraction(extraction, run_index: int = 0):
    """Render a review widget for a single extraction run."""
    if not extraction:
        print("No extraction to review.")
        return

    header = widgets.HTML(
        f"<b>Human Review — Run #{run_index}</b>"
        f"&nbsp;&nbsp;Confidence: <code>{extraction.confidence_score:.2f}</code>"
    )
    kw_out = widgets.Textarea(
        value=_extraction_to_text(extraction),
        description="Keywords:",
        disabled=True,
        layout=widgets.Layout(width="98%", height="180px"),
    )
    sql_out = widgets.Textarea(
        value=getattr(extraction, "sql_query", ""),
        description="SQL Draft:",
        disabled=True,
        layout=widgets.Layout(width="98%", height="110px"),
    )
    label_btn = widgets.ToggleButtons(
        options=["good", "partial", "bad"],
        value="partial",
        description="Label:",
        style={"button_width": "90px", "description_width": "50px"},
    )
    notes_in = widgets.Textarea(
        placeholder="What was correct / wrong? Missing context? SQL issues?",
        description="Notes:",
        layout=widgets.Layout(width="98%", height="70px"),
    )
    submit_btn = widgets.Button(description="Submit Review", button_style="primary", icon="check")
    feedback_out = widgets.Output()

    def on_submit(_):
        human_reviews.append(
            {
                "timestamp": datetime.now().isoformat(timespec="seconds"),
                "run_index": run_index,
                "label": label_btn.value,
                "notes": notes_in.value.strip(),
                "sql_query": getattr(extraction, "sql_query", ""),
                "confidence": round(extraction.confidence_score, 3),
            }
        )
        with feedback_out:
            clear_output()
            print(f"Saved: label='{label_btn.value}' | session total: {len(human_reviews)}")

    submit_btn.on_click(on_submit)
    display(
        widgets.VBox(
            [header, kw_out, sql_out, label_btn, notes_in, submit_btn, feedback_out],
            layout=widgets.Layout(border="1px solid #ddd", padding="12px"),
        )
    )


# Auto-launch for the most recent agent run
review_extraction(lg_result.get("extraction"), run_index=len(session_metrics))

In [ ]:

# ── Session Summary: Metrics + Human Eval Report ─────────────────────────────
# Run this cell at any time to print the full observability picture:
#   • Latency, token usage, and error rate across all LLM calls this session
#   • Human label breakdown (good / partial / bad)

from collections import Counter


def show_session_summary():
    # ── LLM Metrics ──────────────────────────────────────────────────────────
    print("━━━━━━━━━━━━━━━━━━━━━━ OBSERVABILITY METRICS ━━━━━━━━━━━━━━━━━━━━━━")
    if not session_metrics:
        print("  No LLM calls recorded yet — run the agent above first.")
    else:
        errors = [m for m in session_metrics if m.error]
        ok = [m for m in session_metrics if not m.error]
        print(f"  {'#':<4} {'Timestamp':<22} {'ms':>7}  {'Prompt':>8}  {'Compl':>6}  {'Total':>7}  Status")
        print(f"  {'-' * 64}")
        for i, m in enumerate(session_metrics, 1):
            status = f"ERR: {m.error[:30]}" if m.error else "ok"
            print(
                f"  {i:<4} {m.timestamp:<22} {m.latency_ms:>7.0f}"
                f"  {m.prompt_tokens:>8}  {m.completion_tokens:>6}  {m.total_tokens:>7}  {status}"
            )
        print(f"\n  Total calls  : {len(session_metrics)}")
        print(f"  Error rate   : {len(errors)}/{len(session_metrics)} ({len(errors) / len(session_metrics) * 100:.0f}%)")
        if ok:
            avg_ms = sum(m.latency_ms for m in ok) / len(ok)
            print(f"  Avg latency  : {avg_ms:.0f} ms")
        print(f"  Total tokens : {sum(m.total_tokens for m in session_metrics):,}")

    # ── Human Eval ───────────────────────────────────────────────────────────
    print("\n━━━━━━━━━━━━━━━━━━━━━━ HUMAN EVAL SUMMARY ━━━━━━━━━━━━━━━━━━━━━━━━")
    if not human_reviews:
        print("  No reviews yet — submit labels using the widget above.")
    else:
        counts = Counter(r["label"] for r in human_reviews)
        total = len(human_reviews)
        print(f"  Total reviews : {total}")
        for label in ["good", "partial", "bad"]:
            n = counts.get(label, 0)
            bar = "█" * n + "░" * (total - n)
            print(f"  {label:8s}: {bar}  {n}/{total} ({n / total * 100:.0f}%)")
        print("\n  Recent entries:")
        for r in human_reviews[-5:]:
            note = f'  "{r["notes"]}"' if r["notes"] else ""
            print(f"    [{r['timestamp']}] Run #{r['run_index']} → {r['label']}{note}")


show_session_summary()


## 5. SQL Explorer
Edit the `sql` variable and re-run this cell to query the in-memory DB.
The connection (`conn`) was created in the cell above and stays open for the lifetime of the kernel.

In [ ]:
def run_query(sql):
    cur = conn.cursor()
    try:
        cur.execute(sql)
        rows = cur.fetchall()
        col_names = [d[0] for d in cur.description] if cur.description else []
        print(' | '.join(col_names))
        print('-' * 60)
        for row in rows:
            print(' | '.join(str(v) for v in tuple(row)))
        print()
        print(str(len(rows)) + ' row(s)')
    except Exception as e:
        print('SQL Error: ' + str(e))


# --- Change the query below and re-run this cell ---
sql = """
SELECT name, condition, threshold, points, description
FROM   rules
WHERE  policy_id = 'silverpeak-pol'
ORDER  BY points DESC
"""

# Other example queries (uncomment to try):
# sql = "SELECT * FROM employees ORDER BY points DESC"
# sql = "SELECT * FROM policies"
# sql = "SELECT * FROM organizations"
# sql = "SELECT e.first_name, e.last_name, e.points, r.name AS region FROM employees e JOIN regions r ON e.region_id = r.id"
# sql = "SELECT name FROM sqlite_master WHERE type='table'"

run_query(sql)


In [ ]:
sql = "SELECT * FROM employees ORDER BY points DESC"
run_query(sql)